In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from deeporigin.drug_discovery import  BRD_DATA_DIR, Protein, Ligand, LigandSet, ABFE, SystemPrep, ABFEParams, PreparedSystem
from deeporigin.platform import DeepOriginClient

client = DeepOriginClient()


# ABFE workflow

This notebook shows you how to run ABFE on Deep Origin, and serves as a quick tutorial for running ABFE. 


In [ ]:
ligands = LigandSet.from_dir(BRD_DATA_DIR)
ligands.to_dataframe()

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.sync()
protein.show()

In [ ]:
# use this ligand
ligand = [ligand for ligand in ligands if ligand.name == "cmpd 4 (Crotyl)"][0]
ligand.sync()
ligand

## Prepare system

In [ ]:
# see if we have prepared systems for this. if now, prepare one
systems = PreparedSystem.from_result(protein_id=protein.id)
sp = SystemPrep(protein=protein, ligand=ligand)

if len(systems) > 0:
    system = systems[0]
else:
    system = sp.run()
    
system.show()



## Quote ABFE

We can estimate the cost of an ABFE run without running it. 

In [ ]:
abfe = ABFE(prepared_system=system, params=ABFEParams(test_run=0))


In [ ]:
abfe

In [ ]:
abfe.quote()
abfe.estimate

If we're happy with this price, we can confirm the job (which runs it)

In [ ]:
abfe.start()

In [ ]:
task = await abfe.watch()

In [ ]:
abfe.get_results()

In [ ]:
abfe.sync()
abfe.progress

In [ ]:
client.entities.search_ligands(limit=10)

## Monitor job

To monitor a job, use the job.watch method:

In [ ]:
abfe.cancel()

## Get results

In [ ]:
abfe.get_results()


In [ ]:
client.files.list("entities/ligands/")

In [ ]:
client.files.download("tool-runs/468308fb-2d1f-415b-b2d9-2b0f9b5fdd73/protein/ligand/binding/binding/window_2/Prod_1/solute_trajectory_20ps.xtc")

In [ ]:
client.files.list("tool-runs/468308fb-2d1f-415b-b2d9-2b0f9b5fdd73/protein/ligand/binding/binding/window_2/Prod_1/")

In [ ]:
abfe = ABFE.from_id("5cab7747-034f-4c4e-9bd3-232b0282917a")
abfe

In [ ]:
abfe.get_results()

In [ ]:
abfe.show_convergence_time()

In [ ]:
abfe.show_overlap_matrix()

In [ ]:
abfe.show_trajectory(step="binding", window=2)

In [ ]:
from deeporigin_molstar.src.viewers import ProteinViewer

protein_viewer = ProteinViewer(
    data="/Users/srinivas/.deeporigin/tool-runs/0b25519f-bbb5-4a03-9129-5c0f03ef80a6/system.pdb", format="pdb"
)
html_content = protein_viewer.render_trajectory("/Users/srinivas/.deeporigin/tool-runs/5cab7747-034f-4c4e-9bd3-232b0282917a/protein/ligand/binding/binding/window_2/Prod_1/solute_trajectory_20ps.xtc")

from deeporigin_molstar import JupyterViewer

JupyterViewer.visualize(html_content)

In [ ]:
from deeporigin.drug_discovery import Protein
protein = Protein.from_file("/Users/srinivas/.deeporigin/tool-runs/0b25519f-bbb5-4a03-9129-5c0f03ef80a6/system.pdb")
protein.remove_water()
protein.remove_hetatm()
protein.show()

In [ ]:
from deeporigin_molstar.src.viewers import ProteinViewer

protein_viewer = ProteinViewer(
    data=protein.to_file(), format="pdb"
)
html_content = protein_viewer.render_trajectory("/Users/srinivas/.deeporigin/tool-runs/5cab7747-034f-4c4e-9bd3-232b0282917a/protein/ligand/binding/binding/window_2/Prod_1/solute_trajectory_20ps.xtc")

from deeporigin_molstar import JupyterViewer

JupyterViewer.visualize(html_content)